In [1]:
from IPython.display import HTML
HTML("<style>.container{width:100%!important;margin:auto}div.cell,div.input_area{padding:2px;margin:0}div.output_wrapper{padding:0;margin:0}</style>")

# RISE Data Science Practicum
## Week 1, Day 3 — NumPy and Pandas

**Goal for today:** learn the two libraries that almost everything else in this course is built on top of. NumPy gives you fast, efficient arrays of numbers. Pandas builds on NumPy to give you labeled, spreadsheet-like tables — the data structure you'll use constantly for the rest of the summer.

**Running example:** We'll continue with our **weather station** theme from Day 2, but today it's a real dataset: `station_weather.csv`, containing readings from three stations (RISE-01, RISE-02, RISE-03) across three days.

### How to use this notebook
- Run each cell in order (Shift + Enter in Colab).
- **Before you start:** upload `station_weather.csv` to your Colab session/local drive. You'll need it partway through this notebook.

---


## 1. NumPy Basics

NumPy ("Numerical Python") is a library for working with **arrays** — grids of numbers, all of the same type. It's the foundation that pandas, scikit-learn, and most of the data science world is built on.


### Why NumPy instead of plain Python lists?

Yesterday, you wrote loops like this to compute an average:

```python
total = 0
for temp in temperatures:
    total += temp
average = total / len(temperatures)
```

That works fine for 7 numbers. But real datasets can have millions of numbers. Python's regular `for` loops are slow for this because Python checks the type of every single element, one at a time, as it loops.

NumPy arrays solve this by:
1. Storing all values as the **same type**, packed tightly in memory (no per-element type-checking).
2. Using **vectorized operations** — operations that apply to an entire array at once, implemented in fast, compiled C code under the hood instead of slow Python loops.

Let's see the difference directly.


In [1]:
import numpy as np      # "np" is the near-universal alias — always use it
import time

# Compare: summing a list of 1 million numbers, Python loop vs NumPy
n = 1_000_000
python_list = list(range(n))
numpy_array = np.arange(n)

# Python loop
start = time.time()
total = 0
for x in python_list:
    total += x
python_time = time.time() - start

# NumPy vectorized operation
start = time.time()
# TODO: use the NumPy array's .sum() method to compute the total, instead of a loop
total_np = None
numpy_time = time.time() - start

print(f"Python loop time: {python_time:.4f} seconds")
print(f"NumPy time:       {numpy_time:.4f} seconds")
print(f"NumPy was about {python_time / numpy_time:.0f}x faster")


Python loop time: 0.0719 seconds
NumPy time:       0.0000 seconds
NumPy was about 4310x faster


The exact speedup varies by machine, but NumPy is almost always **dramatically** faster for numerical work — often 10-100x. This matters a lot once your datasets grow beyond toy examples.


### Creating arrays

There are several ways to create a NumPy array. The most common is converting a list.


In [ ]:
# TODO: create a NumPy array from this list using np.array()
temperatures = np.array([4,5,6,7])

print(temperatures)
print(type(temperatures))
print(temperatures.dtype)    # the data type stored in the array
print(temperatures.shape)    # the array's dimensions


In [ ]:
# Other handy ways to create arrays
# TODO: fill in each line using the function named in the comment
zeros = np.zeros(5)                  # array of 5 zeros -- use np.zeros
ones = np.ones(5)                   # array of 5 ones -- use np.ones
sequence = np.arange(0,20,2)                # like range(), but returns an array: start=0, stop=20, step=2 -- use np.arange
evenly_spaced = np.linspace(0,1,5)           # 5 evenly spaced points between 0 and 1 (inclusive) -- use np.linspace

print("zeros:", zeros)
print("ones:", ones)
print("sequence:", sequence)
print("evenly_spaced:", evenly_spaced)


###  Try It
Create a NumPy array called `hours` containing the values `6, 8, 10, 12, 14, 16, 18, 20` using `np.arange`. Print it along with its `.shape`.


In [ ]:
# Your code here



### Selecting parts of an array

Indexing a NumPy array works a lot like indexing a Python list — but slicing is even more powerful, and works the same way across multiple dimensions (we'll keep things to 1D today, but this matters a lot once we hit images and matrices later in the course).


In [24]:
temperatures = np.array([64.0, 66.5, 70.2, 74.8, 77.1, 75.6, 71.3])

# TODO: fill in each line so the printed result matches the comment
print("First element:", temperatures[0])            # should print 64.0
print("Last element:", temperatures[-1])              # should print 71.3, use a negative index
print("First three:", temperatures[0:3])               # should print [64.  66.5 70.2]
print("Every other element:", temperatures[::2])       # should print [64.  70.2 77.1 71.3], use step slicing
print("Reversed:", temperatures[::-1])                  # should print the array backwards


First element: 64.0
Last element: 71.3
First three: [64.  66.5 70.2]
Every other element: [64.  70.2 77.1 71.3]
Reversed: [71.3 75.6 77.1 74.8 70.2 66.5 64. ]


### Vectorized operations

This is the single biggest mental shift from Day 2: instead of looping over each value, you apply an operation to the **whole array at once**.


In [26]:
temperatures_f = np.array([64.0, 66.5, 70.2, 74.8, 77.1, 75.6, 71.3])

# TODO: convert the entire array from Fahrenheit to Celsius — no loop needed!
# formula: (F - 32) * 5 / 9
temperatures_c = (temperatures_f - 32) * 5 / 9
print(temperatures_c)

# Arithmetic between two arrays happens element-by-element
daily_adjustment = np.array([1.0, 1.0, 0.5, 0.5, -0.5, -1.0, -1.0])
# TODO: add daily_adjustment to temperatures_f
adjusted = daily_adjustment + temperatures_c
print(adjusted)


[17.77777778 19.16666667 21.22222222 23.77777778 25.05555556 24.22222222
 21.83333333]
[18.77777778 20.16666667 21.72222222 24.27777778 24.55555556 23.22222222
 20.83333333]


Compare that to the Day 2 version, which needed a full loop:
```python
temperatures_c = []
for t in temperatures_f:
    temperatures_c.append((t - 32) * 5 / 9)
```
Same result, but the NumPy version is shorter, clearer, *and* faster.


### Boolean masks — filtering arrays

A **boolean mask** is an array of `True`/`False` values, usually created by applying a comparison to an array. You can then use that mask to pull out only the values that match.


In [29]:
temperatures_f = np.array([64.0, 66.5, 70.2, 74.8, 77.1, 75.6, 71.3])

# Step 1: create the mask -- TODO: True where temperature is greater than 72
is_hot = (temperatures_f > 72)
print(is_hot)

# Step 2: use the mask to filter the array
# TODO: use is_hot to select only the hot temperatures from temperatures_f
hot_temps = temperatures_f[is_hot]
print(hot_temps)


[False False False  True  True  True False]
[74.8 77.1 75.6]


In [32]:
# You can also do this in a single line
# TODO: filter temperatures_f for values greater than 72, in one line
hot_temps = temperatures_f[temperatures_f > 72]
print(hot_temps)

# Combine conditions with & (and) / | (or) — note: use parentheses around each condition!
# TODO: filter for temperatures between 65 and 75 (inclusive on both ends)
mild_temps = temperatures_f[(temperatures_f >= 65) & (temperatures_f <= 75)]
print(mild_temps)

# Count how many values match a condition
# TODO: use np.sum() on a boolean condition to count hot readings
print("Number of hot readings:", np.sum(mild_temps))


[74.8 77.1 75.6]
[66.5 70.2 74.8 71.3]
Number of hot readings: 282.8


 **Common mistake:** Python's regular `and`/`or` keywords don't work element-by-element on arrays — you must use `&` and `|`, with parentheses around each condition, e.g. `(a > 1) & (b < 5)`.


###  Try It
Using `hours` and `temperatures_f` below, create a boolean mask for readings taken **after hour 12** (i.e., afternoon/evening), then use it to print only those temperatures.


In [ ]:
hours = np.array([6, 8, 10, 12, 14, 16, 18, 20])
temperatures_f = np.array([58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4])

# Your code here



### Summary functions

NumPy arrays come with built-in methods for common statistics — no manual loop needed.


In [34]:
temperatures_f = np.array([58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4])

# TODO: fill in using the method named in each comment
print("Mean:", temperatures_f.mean())                  # .mean()
print("Standard deviation:", temperatures_f.std())    # .std()
print("Sum:", temperatures_f.sum())                   # .sum()
print("Min:", temperatures_f.min())                   # .min()
print("Max:", temperatures_f.max())                   # .max()

# You can also call these as functions instead of methods — both work the same:
print("Mean (function form):", np.mean(temperatures_f))  # np.mean(temperatures_f)


Mean: 69.5
Standard deviation: 6.244797834998342
Sum: 556.0
Min: 58.2
Max: 77.8
Mean (function form): 69.5


 **Worked Example:** finding *where* the extreme values are, not just their values


In [16]:
import math
import numpy as np

temperatures_f = np.array([58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4])
hours = np.array([6, 8, 10, 12, 14, 16, 18, 20])

#mean 
mean = temperatures_f.mean()
print(mean)
#standard deviation
st_dev = np.sqrt((((temperatures_f-mean)**2).sum())/temperatures_f.size)
print(st_dev)
#MAD 
mad = (np.absolute(temperatures_f-mean)).sum()/temperatures_f.size
print(mad)
#MedianAD
median = np.quantile(temperatures_f,0.5)
MedAD = np.quantile((np.absolute(temperatures_f-median)), 0.5)
print(MedAD)
#IQR
iqr = (np.quantile(temperatures_f,0.75) - np.quantile(temperatures_f,0.25))/2
print(iqr)
# argmax / argmin return the INDEX of the max/min value, not the value itself
# TODO: get the index of the hottest and coolest readings
hottest_index = np.argmax(temperatures_f)    # use .argmax()
coolest_index = np.argmin(temperatures_f)    # use .argmin()

print(f"Hottest reading: {temperatures_f[hottest_index]}°F at hour {hours[hottest_index]}")
print(f"Coolest reading: {temperatures_f[coolest_index]}°F at hour {hours[coolest_index]}")


69.5
6.244797834998342
5.274999999999999
5.399999999999999
4.787499999999994
Hottest reading: 77.8°F at hour 14
Coolest reading: 58.2°F at hour 6


###  Try It
Using `temperatures_f`, compute and print:
1. The **range** (max minus min)
2. How many readings are **within 1 standard deviation of the mean** (hint: build a boolean mask using `mean()` and `std()`, then `.sum()` it)


In [ ]:
temperatures_f = np.array([58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4])

# Your code here



---

## 2. Pandas Basics

NumPy is great for arrays of numbers, but real datasets usually have **labels** — column names, row identifiers, mixed data types (text *and* numbers together). That's what **pandas** is for. Pandas is built on top of NumPy, and you'll use it constantly for the rest of this course.

Two core pandas structures:

| Structure | What it is | Think of it as... |
|---|---|---|
| `Series` | a single labeled column of data | one column of a spreadsheet |
| `DataFrame` | a full table of labeled columns | the whole spreadsheet |


### Series — a single labeled column


In [39]:
import pandas as pd     # "pd" is the near-universal alias — always use it

# TODO: create a pandas Series from this list using pd.Series()
temperatures = [58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4]
print(temperatures)
temperatures = pd.Series([58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4])
print(temperatures)
print(type(temperatures))


[58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4]
0    58.2
1    64.0
2    69.3
3    74.1
4    77.8
5    76.2
6    71.0
7    65.4
dtype: float64
<class 'pandas.Series'>


In [40]:
# A Series can have a meaningful index (label) instead of just 0, 1, 2, ...
hours = [6, 8, 10, 12, 14, 16, 18, 20]
# TODO: create a Series with index=hours
temperatures = pd.Series([58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4], index=hours)
print(temperatures)

# Now you can look values up by their meaningful label
print("Temperature at hour 14:", None)   # TODO: look up the value at label 14


6     58.2
8     64.0
10    69.3
12    74.1
14    77.8
16    76.2
18    71.0
20    65.4
dtype: float64
Temperature at hour 14: None


### DataFrame — a full table

A DataFrame is the structure you'll use majority of the time in this course. You can build one from a dictionary of lists — each key becomes a column.


In [42]:
data = {
    "hour": [6, 8, 10, 12, 14, 16, 18, 20],
    "temperature_f": [58.2, 64.0, 69.3, 74.1, 77.8, 76.2, 71.0, 65.4],
    "humidity_pct": [70, 65, 58, 50, 45, 48, 55, 63],
}

# TODO: build a DataFrame from the dictionary above using pd.DataFrame()
df = pd.DataFrame(data)
print(df)
print(type(df))


   hour  temperature_f  humidity_pct
0     6           58.2            70
1     8           64.0            65
2    10           69.3            58
3    12           74.1            50
4    14           77.8            45
5    16           76.2            48
6    18           71.0            55
7    20           65.4            63
<class 'pandas.DataFrame'>


Notice how similar this is to the **list of dictionaries** pattern from Day 2 — a DataFrame is really just a more powerful, more convenient version of that same idea.


### Try It
Build a small DataFrame called `wind_df` with two columns: `"hour"` (use `[6, 8, 10, 12]`) and `"wind_mph"` (use `[6.7, 9.1, 6.3, 7.3]`). Print it.


In [ ]:
# Your code here


---

### Loading data from a file

In practice, you'll almost never type your data in by hand — you'll load it from a file. Pandas makes this nearly effortless.

**Before running the next cell:** make sure `station_weather.csv` has been uploaded to your Colab session (folder icon on the left → upload).


In [43]:
# TODO: load station_weather.csv into a DataFrame using pd.read_csv()
station_df = pd.read_csv("station_weather.csv")
station_df.head()


,date,station_id,hour,temperature_f,humidity_pct,wind_mph
0,2026-06-15,RISE-01,6,59.7,87.3,6.7
1,2026-06-15,RISE-01,8,60.2,93.8,9.1
2,2026-06-15,RISE-01,10,65.0,79.9,6.3
3,2026-06-15,RISE-01,12,71.6,64.1,7.3
4,2026-06-15,RISE-01,14,70.6,61.0,4.8


That one line, `pd.read_csv(...)`, replaces everything you'd need to do manually in plain Python: opening the file, parsing each line, splitting on commas, converting types... pandas handles all of it.

The same pattern works for Excel files:
```python
df = pd.read_excel("filename.xlsx")
```
(this requires the `openpyxl` library to be installed, which Colab already has)


### Try It
Load `station_weather.csv` into a DataFrame called `weather` and display the **last 5 rows** instead of the first (hint: there's a method just like `.head()` for this).


In [ ]:
# Your code here



---

### Inspecting a DataFrame

Before you do *anything* with a new dataset, you should always look it over first. These tools are your starting checklist for any new file.


In [45]:
station_df = pd.read_csv("station_weather.csv")

# TODO: print the shape (rows, columns) of station_df
print("Shape (rows, columns):", station_df.shape)
print()
# TODO: print the column names as a list
print("Column names:", list(station_df.columns))


Shape (rows, columns): (72, 6)

Column names: ['date', 'station_id', 'hour', 'temperature_f', 'humidity_pct', 'wind_mph']


In [46]:
# .info() gives you column types and how many non-missing values each has
# TODO: call .info() on station_df
station_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           72 non-null     str    
 1   station_id     72 non-null     str    
 2   hour           72 non-null     int64  
 3   temperature_f  72 non-null     float64
 4   humidity_pct   71 non-null     float64
 5   wind_mph       71 non-null     float64
dtypes: float64(3), int64(1), str(2)
memory usage: 3.5 KB


 Look closely at the `Non-Null Count` column above — `humidity_pct` and `wind_mph` have **fewer** non-null entries than the other columns. That means there's missing data hiding in this file, just like a real sensor dataset would have. We'll deal with that more later in the course — for now, just know `.info()` is how you'd catch it.


In [49]:
# .describe() gives you quick summary statistics for every numeric column
# TODO: call .describe() on station_df
station_df.describe()


,hour,temperature_f,humidity_pct,wind_mph
count,72.000000,72.000000,71.000000,71.000000
mean,13.000000,65.177778,79.387324,7.008451
std,4.614735,6.870189,11.084673,2.752492
min,6.000000,51.500000,59.600000,0.000000
25%,9.500000,60.450000,68.250000,5.350000
50%,13.000000,65.700000,81.800000,7.300000
75%,16.500000,69.825000,88.300000,8.850000
max,20.000000,79.600000,95.000000,13.700000


### Try It
Run `.info()` and `.describe()` on `station_df`. Based on the output, answer in a comment: which column has missing values, and how many rows have a value in that column?


In [ ]:
station_df = pd.read_csv("station_weather.csv")

# Your code here

# Your answer as a comment:
# 


---

### Selecting data: columns, rows, `.loc`, and `.iloc`

This is where pandas starts to feel like a more powerful version of the lists and dictionaries from Day 2.


In [58]:
station_df = pd.read_csv("station_weather.csv")

# Selecting a single column returns a Series
# TODO: select the "temperature_f" column and call .head() on it
print(station_df["temperature_f"].head())
print(type(station_df["temperature_f"]))


0    59.7
1    60.2
2    65.0
3    71.6
4    70.6
Name: temperature_f, dtype: float64
<class 'pandas.Series'>


In [64]:
# Selecting multiple columns returns a DataFrame — note the double brackets!
# TODO: select both "station_id" and "temperature_f" columns
subset = station_df[["temperature_f", "station_id"]]
subset.head()


,temperature_f,station_id
0,59.7,RISE-01
1,60.2,RISE-01
2,65.0,RISE-01
3,71.6,RISE-01
4,70.6,RISE-01


### `.iloc` — select by **integer position** (just like list indexing)


In [69]:
# .iloc[row_position, column_position] — works like indexing a list or array
# TODO: fill in each line
print(station_df.iloc[0])          # first row, all columns -- station_df.iloc[0]
print()
print(station_df.iloc[0,1])       # first row, second column -- station_df.iloc[0, 1]
print()
print(station_df.iloc[0:3])               # first three rows -- station_df.iloc[0:3]  (just write the expression, no print needed for DataFrame display)


date             2026-06-15
station_id          RISE-01
hour                      6
temperature_f          59.7
humidity_pct           87.3
wind_mph                6.7
Name: 0, dtype: object

RISE-01

         date station_id  hour  temperature_f  humidity_pct  wind_mph
0  2026-06-15    RISE-01     6           59.7          87.3       6.7
1  2026-06-15    RISE-01     8           60.2          93.8       9.1
2  2026-06-15    RISE-01    10           65.0          79.9       6.3 2


### `.loc` — select by **label** (column names, or meaningful row labels)


In [ ]:
# .loc[row_label, column_label]
# TODO: fill in each line
print(None)   # row labeled 0, column "temperature_f" -- station_df.loc[0, "temperature_f"]
print()
None   # rows 0 through 3 (inclusive!), columns "station_id" and "temperature_f" -- station_df.loc[0:3, ["station_id", "temperature_f"]]


 **Subtle but important:** `.iloc` slicing works like Python (end excluded), but `.loc` slicing **includes** the endpoint. `station_df.iloc[0:3]` gives you 3 rows; `station_df.loc[0:3]` gives you 4. This trips up almost everyone at first — keep an eye out for it.


### Filtering rows with conditions

This is the pandas equivalent of NumPy's boolean masks — and it's one of the most useful things you'll do with any dataset.


In [ ]:
station_df = pd.read_csv("station_weather.csv")

# TODO: filter station_df to just rows where station_id equals "RISE-01"
rise01 = None
rise01.head()


In [ ]:
# Combine conditions with & and | — same rule as NumPy: parentheses around each condition!
# TODO: filter for temperature_f > 75 AND humidity_pct < 55
hot_and_dry = None
hot_and_dry


### Worked Example: filter, then summarize

A very common pattern: filter down to the rows you care about, then compute statistics on just that subset.


In [ ]:
station_df = pd.read_csv("station_weather.csv")

# TODO: filter to rows where hour == 14
afternoon = None
print("Average temperature at hour 14, across all stations and days:")
print(afternoon["temperature_f"].mean())

# TODO: filter to rows where station_id == "RISE-02"
rise02_only = None
print()
print("RISE-02 temperature summary:")
print(rise02_only["temperature_f"].describe())


### Try It
Using `station_df`:
1. Use `.loc` to select rows `5` through `10` (inclusive) and just the `"station_id"` and `"humidity_pct"` columns.
2. Filter the DataFrame to rows where `wind_mph` is greater than `10`, and print how many such rows exist (hint: `len(...)` or `.shape[0]`).


In [ ]:
station_df = pd.read_csv("station_weather.csv")

# Your code here (part 1)


# Your code here (part 2)



---

## Putting It All Together

Time to combine NumPy and pandas on the full `station_weather.csv` dataset. Work through these in order.

**Part A.** Load `station_weather.csv` into a DataFrame called `weather`. Print its `.shape` and run `.info()`.

**Part B.** Filter `weather` down to just `"RISE-03"`, and convert its `"temperature_f"` column to a NumPy array using `.to_numpy()` (or `.values`). Use NumPy to print the mean and standard deviation of that array.

**Part C.** Using a boolean mask on the full `weather` DataFrame, find all rows where `temperature_f` is above the **overall dataset mean** (compute the mean first, then filter). Print how many rows match.

**Part D.** Using `.loc`, select just the `"date"`, `"station_id"`, and `"temperature_f"` columns for the **first 5 rows**, and print the result.


In [ ]:
# Part A



In [ ]:
# Part B



In [ ]:
# Part C



In [ ]:
# Part D



---

### Optional Stretch Challenge

Write a function `station_summary(df, station_id)` that takes the full `weather` DataFrame and a station ID string, and returns a dictionary with keys `"avg_temp"`, `"max_temp"`, `"min_temp"`, and `"avg_humidity"` for just that station. Test it on all three stations and print the results.

*(Hint: this combines the function-writing skills from Day 2 with the pandas filtering you just learned today!)*


In [ ]:
def station_summary(df, station_id):
    pass   # replace with your code

# Test it on all three stations
# for sid in ["RISE-01", "RISE-02", "RISE-03"]:
#     print(sid, station_summary(weather, sid))
